# 00 · Exploración de la API de RAWG

## Contexto
Primera fase del pipeline ETL. Exploración inicial de la API de RAWG para entender 
la estructura de los datos disponibles antes de diseñar el esquema de base de datos.

## Objetivo
- Analizar los campos disponibles por videojuego
- Identificar tipos de datos y campos complejos (listas, diccionarios)
- Guardar una muestra de la respuesta para referencia

## Output
Archivo `rawg_sample_response.json` con muestra de 5 videojuegos

In [ ]:
import requests
import json
from datetime import datetime
from time import sleep


In [ ]:
%run ../bootstrap.py

import utils.secrets as s



In [ ]:
print("=== EXPLORACIÓN DE TODOS LOS CAMPOS QUE DEVUELVE LA API DE RAWG (TODOS LOS INDICADORES DISPONIBLES) ===\n")

'''Petición pidiendo sólo 5 juegos.
    - Exploración de la estructura general de la respuesta
    Análisis del primer videojuego:
    - Listado de todos los campos de un videojuego con nombre del campo, tipo de dato y ejemplo
    - Explorar campos complejos (listas o diccionarios)
    Guardar la respuesta en un archivo JSON'''

# Configuración

from utils.secrets import get_rawg_api_key_cached

API_KEY = get_rawg_api_key_cached()
base_url = f"https://api.rawg.io/api/games"

data = []

params = {
    "key": API_KEY,
    "page_size": 5
    }
# Hacer la petición
response = requests.get(base_url, params = params)
    
if response.status_code == 200:
    data = response.json()
        
    print("=== ESTRUCTURA GENERAL DE LA RESPUESTA ===")
        
    print(f"\nClaves principales: {list(data.keys())}\n")
        
    # Información de paginación
    print(f"Total de juegos disponibles: {data.get('count', 'Desconocido')}")
    print(f"Página siguiente: {data.get('next', 'N/A')}")
    print(f"Juegos en esta página: {len(data.get('results', []))}\n")
        
      
    print("=== CAMPOS DISPONIBLES POR VIDEOJUEGO - EXPLORACIÓN DEL PRIMER JUEGO ===")
    # Analizar el primer juego en detalle
    if data['results']:
        first_game = data['results'][0]
            
        print(f"\n EJEMPLO: {first_game.get('name', 'Desconocido')}\n")
            
        # Diccionario para almacenar tipos de datos
        campos_info = {}
            
        for key, value in first_game.items():
            tipo = type(value).__name__
                
            # Mostrar valor de ejemplo (limitar longitud)
            if isinstance(value, (list, dict)):
                ejemplo = f"{tipo} con {len(value)} elemento(s)"
                if isinstance(value, list) and len(value) > 0:
                    ejemplo += f" - Primer elemento: {value[0]}"
                elif isinstance(value, dict):
                    ejemplo += f" - Claves: {list(value.keys())}"
            else:
                ejemplo = str(value)[:50]

            # Mostrar la información del juego en un diccionario: 
            # las claves son cada campo del juego: name, rating, genres... 
            # y los valores son diccionarios que nos informan del tipo de dato de ese campo y un ejemplo del contenido 
            campos_info[key] = {
            'tipo': tipo,
            'ejemplo': ejemplo
            }
                
            print(f"• {key:20} | Tipo: {tipo:10} | Ejemplo: {ejemplo}")

    print("\n Resumen de campos complejos (tipo lista o diccionario)")
                
   
    for key, value in first_game.items():
         # Explorar campos tipo lista
        if isinstance(value, list) and len(value) > 0:
            print(f"\n {key.upper()}:")
            print(f"   Cantidad: {len(value)}")
            print(f"   Primer elemento: {json.dumps(value[0], indent=2)}")
        
        # Explorar campos tipo diccionario   
        elif isinstance(value, dict) and value:
            print(f"\n {key.upper()}:")
            print(f"   {json.dumps(value, indent=2)}")
        
    print("\nGUARDANDO RESPUESTA COMPLETA PARA ANÁLISIS")
    with open('rawg_sample_response.json', 'w', encoding='utf-8') as file:
            json.dump(data, file, indent=2, ensure_ascii=False)
               
    print("Archivo 'rawg_sample_response.json' creado")
    
else:
    print("Error:", response.status_code)